<img src="../assets/ga-logo.png" style="float: left; margin: 20px; height: 55px">

# Clustering with K-Means - Solution

---

### About
Understand what unsupervised learning is, when to apply clustering, and perform a k-means clustering, evaluating cluster quality using Silhouette Coefficient and Inertia.

### Learning Objectives
- Know the difference between supervised and unsupervised learning.
- Understand when to use clustering.
- Perform a k-Means clustering analysis.
- Evaluate clusters for fit with Silhouette Coefficient and Inertia.

### Notebook Guide
- Unsupervised learning
- Introduction to Clustering
- Clustering algorithms
- K-Means Clustering
- K-Means in `scikit-learn`
- Metrics for Assessing Clusters
- How do I know which number to pick for k?
- Bonus Exercise: Old Faithful
- Conclusions and Takeaways

# Unsupervised learning

---

Up until now, we've been doing **supervised learning** - that is, modeling of the form:

> Given X, predict Y


<details>
<summary>Definitions:</summary>

* Regression models predict a **continuous response** (numerical).
* Classification models predict a **discrete response** (categorical).
</details>

<br>

When we don't have a Y variable to predict, we are in the realm of **unsupervised learning**. Since there is no Y variable, unsupervised learning has no measurable "goal". Instead, unsupervised learning seeks to **represent the data in new ways**. Today we're introducing **clustering**; however, there are many other types of unsupervised learning.

> Data without a Y variable are sometimes referred to as **unlabeled** data. This is because the Y variable is often refered to as a **label**.

## Unsupervised learning creates new issues
Since there is no Y variable to "supervise" our learning, unsupervised learning presents us with some new issues we've never had to work through:

* **What is "correct"?** Since there's no Y variable, we don't have an easy to way know if we're even doing a good job.
* **Tuning parameter selection.** Many unsupervised models have tuning parameters. How do we tune them if we don't know how to evaluate our model?
* **Unpredictability (clustering).** In clustering, it is very difficult to predict what our model will give us. It's possible that a clustering algorithm won't give us actionable results. More on this later.

# Introduction to Clustering

---

**Clustering** is a task in which we seek to group our observations in **homogenous clusters**. Since it's unsupervised, it is up to us, the data scientists, to decide what we mean by "homogenous".

## Uses for Clustering
### Marketing
Marketing teams do a lot of data-driven research into who does and does not buy their product (and why). As a marketing data scientist, you might collect demographic information about people in a survey and their spending habits. After clustering, you do some EDA and you might discover:

> People in Cluster A aren't buying our product, but people in Cluster B are. Why?

After some digging, you might make the conclusion and recommendation:

> People in Cluster A have characteristic X, but people in Cluster B do not. In order to sell to Cluster A, we should target our marketing with respect to X.

Maybe X = 
* They don't have cable
* Their political beliefs
* They live in cities

### Political Polling
In the same vein as the above example, instead of buying a product, maybe it's **voting for a certain candidate**.

The popular data-drive journalism website, FiveThirtyEight is doing a lot of research into the upcoming 2020 Democratic primaries. They have done some (subjective!) clustering techniques to divide Democratic voters into five clusters:

* Party Loyalists
* The Left
* Millennials and Friends
* Black voters
* Hispanic voters (sometimes in combination with Asian voters)

They then graded each candidate based on their favorability of these clusters:

<!-- <img src="../assets/imgs/five-corners.png"> -->
![](../assets/five-corners.png)

Their methodology is more speculation than science, but it still provides a good example into how real political polling works. You can read more about it [here](https://fivethirtyeight.com/features/the-5-key-constituencies-of-the-2020-democratic-primary/).

_NOTE:_ Unlike the kinds of clustering we'll see in our class, someone might fall into multiple categories here.

### Recommender Systems
Online retailers cluster their items by similarity. If you buy (or search for) a few items in a given cluster, they may recommend you other similar items in that same cluster.

![](../assets/recs.png)

# Clustering Algorithms 

---

The are many different algorithms that can perform clustering given a dataset. 

These algorithms nearly always reduce to difficult optimization problems which may converge on a local minimum (similarly to gradient descent).

- **K-Means** (mean centroids)
- **DBSCAN** (density based)
- **Hierarchical** (nested clusters by merging or splitting successively)
- **Affinity Propagation** (graph based approach to let points 'vote' on their preferred 'exemplar')
- **Mean Shift** (can find number of clusters)
- **Spectral Clustering**
- **Agglomerative Clustering** (suite of algorithms all based on applying the same criteria/characteristics of one cluster to others)


Image from scikit-learn: ["Comparing different clustering algorithms on toy datasets."](https://scikit-learn.org/stable/auto_examples/cluster/plot_cluster_comparison.html)

![](../assets/sklearn-clustering.png)

From the above, note that $k$-Means clustering:
- is really fast!
- can only create convex clusters. This implies that its clusters can always be linearly separated.
- may have to be run multiple times to get the best clusters.

> - "We argue that there are many clustering algorithms, because the notion of "cluster" cannot be precisely defined."<br>
> - "Therefore, comparing clustering algorithms, must take into account a careful understanding of the inductive principles involved."<br>
> - "The nature of clustering is exploratory, rather than confirmatory."
>
> From: Vladimir Estivill-Castro. ["Why so many clustering algorithms -- A Position Paper" (PDF)](http://web.cs.iastate.edu/~honavar/clustering-survey2.pdf)

Today we're going to look only at one of the algorithms: **k-means**.

# K-Means Clustering

---

#### K-Means is the most popular clustering algorithm

K-means is one of the easier methods to understand and other clustering techniques use some of the same assumptions that k-means relies on.

- **k** is the number of clusters.
- **Means** refers to the mean points of the k clusters.

Similarly to k-nearest neighbors, the resulting centroids partition the space into [Voronoi cells](https://en.wikipedia.org/wiki/Voronoi_diagram). For example:
![](../assets/imgs/sklearn-kmeans-voronoi.png)

Image from scikit-learn: ["A demo of K-Means clustering on the handwritten digits data"](https://scikit-learn.org/stable/auto_examples/cluster/plot_kmeans_digits.html).

**You must choose $k$, the number of clusters, in advance.** Note that this is a huge issue, since we rarely have an intuition for this number. And since we're in unsupervised territory, it's hard for us to know if we've picked correctly! We'll talk about how to pick $k$ later.

The algorithm takes your entire dataset and iterates over its features and observations to determine clusters based around center points. These center points are known as **centroids**. 

**What does K-means do?**

> $K$-means partitions the data into sets of points (clusters). These clusters minimize the within-cluster sum-of-squares.

We will examine this in more detail later!

**K-means iterative fitting:**
1. Pick a value for $k$ (the number of clusters to create).
2. Initialize $k$ 'centroids' (starting points). These do not have to be actual data points!
3. Create clusters by assigning each data point to its nearest centroid.
4. Make your clusters better. Reassign each centroid to the center of its cluster.
5. Repeat steps 3-4 until the centroids converge and do not change across iterations.

$K$-means is guaranteed to converge.

> Check out a demo of this algorithm [here](https://www.naftaliharris.com/blog/visualizing-k-means-clustering/)!


## Initializing Centroids

There are different methods of initializing centroids. For instance:

- Randomly
- Manually
- Special KMeans++ method in Sklearn (_This initializes the centroids to be generally distant from each other_)

**Depending on your problem, you may find some of these are better than others.**

> **Note:** Manual is recommended if you know your data well enough to see the clusters without much help, but rarely used in practice.

## Choosing $k$
This still remains an open question. After all, we're tuning a tuning parameter with no metric for success! Here are three ideas:

* Make an educated guess
    - Industry knowledge (there are five kinds of Democrats...)
    - Visualization (probably impossible if you have more than 2 variables)
* Judge based on a pseudo-evaluation metric, like the **silhouette score**.
* If you're using the resulting cluster labels as input to a supervised learning method later, you can tune $k$ to have the best supervised learning model. This is **transfer learning**.

## A Note on K-Means Convergence

**Knowing your domain and dataset is essential. Evaluating the clusters visually is a must (if possible).**

> _"Given enough time, k-means will always converge, however this may be to a local minimum. This is highly dependent on the initialization of the centroids. As a result, the computation is often done several times, with different initializations of the centroids. One method to help address this issue is the k-means++ initialization scheme, which has been implemented in scikit-learn (use the init='kmeans++' parameter). This initializes the centroids to be (generally) distant from each other, leading to provably better results than random initialization, as shown in the reference."_ [sklearn Clustering Guide](http://scikit-learn.org/stable/modules/clustering.html#k-means)

# K-Means in `scikit-learn`

---

Below we will implement K-Means using `scikit-learn`.

In [ ]:
from sklearn.datasets import make_blobs
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

import numpy as np
import pandas as pd

import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib

matplotlib.style.use('ggplot')

# Let's make some more blobs to test K-Means on
data, color = make_blobs(n_samples=100, random_state=42, centers=3, cluster_std=1.5)

df = pd.DataFrame(data, columns=["x1", "x2"])
df.head()

In [ ]:
# Take a look at the data using a scatterplot
df.plot(kind="scatter", x="x1", y="x2", figsize=(12,6), s=50)
plt.xlabel("$X_1$", fontsize=18)
plt.ylabel("$X_2$", fontsize=18);

In [ ]:
# Define X
X = df[['x1', 'x2']]

In [ ]:
# There's a VERY important step before fitting our K-Means Model. Any ideas?
sc = StandardScaler()
X_scaled = sc.fit_transform(X)

In [ ]:
# Fit a K-means clustering model
km = KMeans(n_clusters=3, random_state=42)
km.fit(X_scaled)

## What did we discover?

### I. Cluster Predictions

In [ ]:
# Option 1: Class attribute

km.labels_

In [ ]:
# Option 2: Predict

km.predict(X_scaled)

In [ ]:
# Attach predicted cluster to original points

df['cluster'] = km.labels_
df.head()

### II. Centroids

In [ ]:
# Option 1: Using groupby

df.groupby('cluster').mean()

In [ ]:
# Option 2: (Preferred method) Using cluster_centers_

km.cluster_centers_

# But wait... why does this not match the one above?!

In [ ]:
# Remember, we scaled this! We need to "unscale".

centroids = sc.inverse_transform(km.cluster_centers_)

## Visually Verifying Cluster Labels

In [ ]:
# Create Centroid DataFrame
centroids = pd.DataFrame(
    centroids,
    columns=["x1", "x2"]
)
centroids

In [ ]:
# Create Scatterplot!

# Figsize
plt.figure(figsize=(10,8))

# Map colors for different clusters
colors = ["red", "green", "blue"]
df['color'] = df['cluster'].map(lambda p: colors[p])

# Plot points
ax = df.plot(    
    kind="scatter", 
    x="x1", y="x2",
    figsize=(10,8),
    c = df['color']
)

# Plot Centroids
centroids.plot(
    kind="scatter", 
    x="x1", y="x2", 
    marker="*", c=["r", "g", "b"], s=550, ax=ax
);

# Metrics for Assessing Clusters
---
## Inertia

Sum of squared differences between each point in a cluster and that cluster's centroid.

How dense is each cluster? 


- low inertia = dense cluster
- ranges from 0 to very high values


$$ \sum_{j=0}^{n} (x_j - \mu_i)^2 $$

where $\mu_i$ is a cluster centroid



`.inertia_` is an attribute of a fitted sklearn's kmeans object

#### Lower inertia is better! Though be cautious with adding too many clusters.

#### Get the inertia

In [ ]:
km.inertia_

---
## Silhouette Score

Tells you how much closer data points are to their own clusters than to the nearest neighbor cluster.

How far apart are the clusters?
- ranges from -1 to 1
- high silhouette score means the clusters are well separated



### $s_i = \frac{b_i - a_i}{max\{a_i, b_i\}}$

Where:
- $a_i$ = Cohesion: Mean distance of points within a cluster from each other.
- $b_i$ = Separation: Mean distance from point $x_i$ to all points in the next nearest cluster.

Use scikit-learn: `metrics.silhouette_score(X_scaled, labels)`.

#### Higher silhouette score is better!


#### Compute the silhouette score 

In [ ]:
from sklearn.metrics import silhouette_score

In [ ]:
silhouette_score(X_scaled, km.labels_)

# How do I know which number to pick for k? 🤔

Finding the best k to use for k-means clustering is not a simple task.
  

NO ground truth, so there isn't necessarily a "correct" number of clusters. 

The business application is often an important consideration:
   - e.g. I need to create 3 profiles for marketing to target
   - Even if the most natural-looking clusters are small, we may try to group several of them together so that it makes financial sense to target those groups.

**Other common approaches:**
- Previous experience.
- Using the elbow method to find a number of clusters that no longer seems to improve a clustering metric by a noticeable degree.
- Scatter plots show separable clusters
- The silhouette coefficient 
- Inertia
- If we're using clustering to improve performance on a supervised learning problem, then we can use our usual methods to test predictions and tune k.
 

## Use the elbow method to choose _k_

The [elbow method](https://en.wikipedia.org/wiki/Elbow_method_(clustering)) is one possible method to help narrow in on the ideal value of **K**. 

### Choose the number of clusters where the next cluster doesn't significantly improve performance. 

#### Let's use the `.interia_` method as our evaluation metric

In [ ]:
inertia_list = []

for k in range(1, 11):
    kmeans = KMeans(n_clusters=k, random_state=42)
    kmeans.fit(X_scaled)
    inertia_list.append(kmeans.inertia_)
    
inertia_list

#### Analyze the above

- Do you see the **"elbow"**?

- What's the best value for _k_?



In [ ]:
plt.plot(range(1, 11), inertia_list, marker='o')
plt.xlabel('# of Clusters')
plt.ylabel('Score')
plt.title('Inertia Scores');


#### Now let's make a plot with the silhouette score

## Remember: Higher is Better for Silhouette Score!

In [ ]:
silhouette_list = []

for k in range(2, 11):
    kmeans = KMeans(n_clusters=k, random_state=42)
    kmeans.fit(X_scaled)
    silhouette_list.append(silhouette_score(X_scaled, kmeans.labels_))
silhouette_list

In [ ]:
plt.plot(range(2, 11), silhouette_list, marker='o')
plt.xlabel('# of Clusters')
plt.ylabel('Score')
plt.title('Silhouette Scores');

#### What _k_ should we choose?

3!

# Bonus Exercise 🏋️‍♂️🏋️‍♀️ : Old Faithful
---
The classic Old Faithful dataset describes the durations of eruptions and the amount of waiting time since the last eruption for the Old Faithful geyser. Plot this data. Do you think this data could benefit from clustering? How many clusters are there?

In [ ]:
df = pd.read_csv("../data/geyser.csv")
df.head()

In [ ]:
plt.scatter(df['duration'], df['waiting'], c="black", s=10);

In [ ]:
# Define X
X = df[['duration', 'waiting']]

In [ ]:
# There's a VERY important step before fitting our K-Means Model. Any ideas?
sc = StandardScaler()
X_scaled = sc.fit_transform(X)

In [ ]:
# Fit a K-means clustering model
km = KMeans(n_clusters=2, random_state=42)
km.fit(X_scaled)

In [ ]:
df['cluster'] = km.labels_
df.head()

In [ ]:
# Create Centroid DataFrame
centroids = pd.DataFrame(
    sc.inverse_transform(km.cluster_centers_),
    columns=["duration", "waiting"]
)
centroids

In [ ]:
# Create Scatterplot!

# Figsize
plt.figure(figsize=(10,8))

# Map colors for different clusters
colors = ["red", "blue"]
df['color'] = df['cluster'].map(lambda p: colors[p])

# Plot points
ax = df.plot(    
    kind="scatter", 
    x="duration", y="waiting",
    figsize=(10,8),
    c = df['color']
)

# Plot Centroids
centroids.plot(
    kind="scatter", 
    x="duration", y="waiting", 
    marker="*", c=["r", "b"], s=550, ax=ax
);

In [ ]:
plt.plot(range(1, 11), inertia_list, marker='o')
plt.xlabel('# of Clusters')
plt.ylabel('Score')
plt.title('Inertia Scores');

In [ ]:
silhouette_list = []

for k in range(2, 11):
    kmeans = KMeans(n_clusters=k, random_state=42)
    kmeans.fit(X_scaled)
    silhouette_list.append(silhouette_score(X_scaled, kmeans.labels_))
silhouette_list

In [ ]:
plt.plot(range(2, 11), silhouette_list, marker='o')
plt.xlabel('# of Clusters')
plt.ylabel('Score')
plt.title('Silhouette Scores');

### How many clusters should be used?

2!

# Conclusions and Takeaways
---

Remember that different starting points may give you different clusters. ⚠️ You won't necessarily get an optimal set of clusters!    

You've learned about when and how to use the unsupervised learning clustering algorithm _k-means_. You saw how to save your cluster labels in a DataFrame. You learned several way to evaluate the best number of clusters. 

### Questions...

- What's the main difference between supervised learning and unsupervised learning?
- Why use clustering?
- What does k-means clustering do?
- What does silhouette score measure?

## More Unsupervised learning in the wild

> 1. www.storyblocks.com -- "More Like This" 
> 1. www.amazon.com
> 1. Faces in your photos 

<img src= 'https://support.apple.com/library/content/dam/edam/applecare/images/en_US/iOS/ios15-iphone12-pro-photos-people-album-favorites.png' style='height:500px' />


## Further reading

- The [scikit-learn documentation](http://scikit-learn.org/stable/modules/clustering.html) has a great summary of many other clustering algorithms.
- This [PyData talk](https://www.youtube.com/watch?v=Mf6MqIS2ql4) is good overview of clustering, different algorithms, and how to think about the quality of your clusters.